# 20 · Event Grid: un broker MQTT che si fida dei certificati di ADR

Finora il certificato che ADR emette a un device l'ha usato solo l'IoT Hub. Qui lo usiamo **da un'altra parte**: un **broker MQTT di Azure Event Grid**.
Event Grid sa fidarsi di una CA che gli si carica: gli carichiamo la **Issuing CA di ADR**, e i device con un certificato di ADR possono presentarsi.

Il notebook crea il namespace con MQTT acceso, carica la CA, e **censisce** un device come client. Il notebook 21 fa collegare il device.

**Due device fatti apposta**, con il nome che dice a cosa servono. Non sono robot né camion della flotta:
- `eventgrid-sensore-01`: lo censiamo come client di Event Grid, in **due modi** (per CA e per impronta, qui sotto);
- `eventgrid-non-censito-01`: ha anche lui un certificato di ADR, ma **non** lo censiamo. Serve al notebook 21 come controprova.
Sono device veri: si registrano a DPS e ricevono il certificato da ADR, come gli altri. Li registra il notebook, se non ci sono già.

**Serve:** i notebook 01 e 02a-02b già eseguiti (il namespace ADR, DPS e la CA di bootstrap del gruppo `robots`). I due device li registra questo notebook.
**Costo:** un namespace Event Grid. Sta nello stesso resource group: il notebook 99 lo elimina insieme al resto.

**Da dire.** "Il certificato di ADR è un certificato X.509 normale. L'Hub non è l'unico posto dove vale: basta che l'altro servizio si fidi della stessa CA."

In [ ]:
# Carica la libreria e legge lo scenario (notebooks\_scenario.psd1): da qui in poi si usa $cfg
. ([scriptblock]::Create([IO.File]::ReadAllText((Get-ChildItem . , .\notebooks , .. -Filter _start.ps1 -ErrorAction SilentlyContinue | Select-Object -First 1).FullName)))

Fabric: valori dalla Eventstream creata dal 17a (417bd75b-4747-4eff-9d6e-d00ca5964001)

Scenario: rg-iot26-dev-uks-001 | namespace ADR adrns-iot-dev-uks | output in D:\Dev\2026\IoTHub\WPC2026\artifacts

In [ ]:
if (-not $cfg -or -not $out) { throw "Mancano le variabili dello scenario: esegui prima la cella di caricamento (la prima cella di codice) di questo notebook." }
$eg = $cfg.EventGridName                       # il nome del namespace Event Grid (dallo scenario)
$sensore = 'eventgrid-sensore-01'              # il device che censiamo come client di Event Grid: deve entrare
$nonCensito = 'eventgrid-non-censito-01'       # ha un certificato di ADR ma NON lo censiamo: serve al notebook 21
$sensoreImpronta = "$sensore-impronta"        # lo stesso device, censito per impronta del certificato
$clients = $sensore                            # i device che censiamo come client di Event Grid
"Event Grid: $eg | device fatti apposta: $sensore (censito), $nonCensito (non censito)"


Event Grid: evgns-iot26-dev-uks-001 | certificato di robot-01 in: D:\Dev\2026\IoTHub\WPC2026\artifacts\devices\robot-01\20261117-015619

## Due device fatti apposta
Prima di tutto servono i due device. Ognuno si **registra a DPS** con il certificato di bootstrap della flotta `robots` e riceve da ADR il suo certificato: sono i passi del notebook 02c.
Se un device ha già il certificato (perché hai rieseguito il notebook) la cella lo riusa: non ne emette un secondo.


In [ ]:
$cartella = Initialize-DemoDevice -Device ($customDevices | Where-Object Id -eq $sensore) -OutDir $out -DpsName $dps.Name -ResourceGroup $rg -Namespace $adrName
$null = Initialize-DemoDevice -Device ($customDevices | Where-Object Id -eq $nonCensito) -OutDir $out -DpsName $dps.Name -ResourceGroup $rg -Namespace $adrName
"Certificato di $sensore in: $cartella"


Event Grid: evgns-iot26-dev-uks-001 | certificato di robot-01 in: D:\Dev\2026\IoTHub\WPC2026\artifacts\devices\robot-01\20261117-015619

## 1 · Il namespace con MQTT
Un namespace Event Grid con **MQTT acceso** (`--topic-spaces-configuration state=Enabled`). La cella salta la creazione se il namespace esiste già.

In [ ]:
$esiste = az eventgrid namespace show --name $eg --resource-group $rg --query name -o tsv --only-show-errors 2>$null
if ($esiste) { "Il namespace $eg esiste gia'." }
else {
    az eventgrid namespace create `
        --name $eg `
        --resource-group $rg `
        --location $rgRegion `
        --topic-spaces-configuration state=Enabled `
        --tags workload=$workload environment=$stage `
        -o none `
        --only-show-errors
    Assert-Az "Creazione del namespace Event Grid $eg"
    "Namespace $eg creato."
}

Namespace evgns-iot26-dev-uks-001 creato.

Error: C:\Users\marco\.azure\cliextensions\eventgrid\azext_eventgrid\aaz\latest\eventgrid\namespace\client\_create.py:54: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
  pattern="^[-a-zA-Z0-9:\._]*$",
C:\Users\marco\.azure\cliextensions\eventgrid\azext_eventgrid\aaz\latest\eventgrid\namespace\client\_delete.py:56: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
  pattern="^[-a-zA-Z0-9:\._]*$",
C:\Users\marco\.azure\cliextensions\eventgrid\azext_eventgrid\aaz\latest\eventgrid\namespace\client\_show.py:54: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
  pattern="^[-a-zA-Z0-9:\._]*$",
C:\Users\marco\.azure\cliextensions\eventgrid\azext_eventgrid\aaz\latest\eventgrid\namespace\client\_upd

L'indirizzo del broker MQTT. Il nome della proprietà non è nella documentazione che ho letto, quindi la cella la cerca nella risposta e, se non la trova, la stampa per intero.

In [ ]:
$ns = az eventgrid namespace show --name $eg --resource-group $rg -o json --only-show-errors | ConvertFrom-Json
Assert-Az "Lettura del namespace $eg"
$mqttHost = $ns.topicSpacesConfiguration.hostname
if (-not $mqttHost) { $ns | ConvertTo-Json -Depth 6; throw "Non trovo l'indirizzo MQTT: guarda la risposta qui sopra (topicSpacesConfiguration)." }
"Broker MQTT: $mqttHost  (porta 8883, TLS)"

## 2 · La CA di ADR
Event Grid vuole la CA che è il **genitore diretto** del certificato del device: per ADR è la **Issuing CA**, la seconda del file `issued-chain.pem` che il device ha ricevuto.
È una CA: la sua parte pubblica non è un segreto. La cella la estrae dalla catena di `eventgrid-sensore-01` e la carica.

In [ ]:
$ca = Export-AdrCaCertificates -ChainPath (Join-Path $cartella 'issued-chain.pem') -OutDir (Join-Path $out 'eventgrid')
"Issuing CA di ADR: $($ca.IcaSubject)"
az eventgrid namespace ca-certificate create `
    --name ca-adr-ica `
    --namespace-name $eg `
    --resource-group $rg `
    --certificate $ca.Ica `
    -o none `
    --only-show-errors
Assert-Az "Caricamento della CA di ADR in $eg"
"CA caricata come ca-adr-ica."

## 3 · Censire il device, per CA
Event Grid ha il **suo registro di client**. Un device è un client con un **nome di autenticazione**. Con `SubjectMatchesAuthenticationName` il nome deve essere uguale al
nome del certificato, cioè al **registration ID** del device (`CN=eventgrid-sensore-01`): il certificato dice chi sei, il client dice che ti conosciamo.
È il modo adatto a una flotta: non si elenca nessun certificato, basta che la CA sia quella giusta.


In [ ]:
foreach ($c in $clients) {
    az eventgrid namespace client create `
        --name $c `
        --namespace-name $eg `
        --resource-group $rg `
        --authentication-name $c `
        --client-certificate-authentication validation-scheme=SubjectMatchesAuthenticationName `
        --description "Device registrato in ADR: $c" `
        -o none `
        --only-show-errors
    Assert-Az "Creazione del client $c"
    "$c censito."
}

## 4 · Censire lo stesso device, per impronta
Il secondo modo non guarda la catena: confronta l'**impronta** (thumbprint) del certificato con quella scritta nel client. Il nome del client è libero.
Lo aggiungiamo perché nel notebook 21 si vede che i due modi non danno lo stesso risultato con un certificato di ADR.
Il prezzo di questo modo: a ogni rinnovo il certificato cambia, quindi cambia l'impronta e il client va aggiornato.


In [ ]:
$leaf = [regex]::Match([IO.File]::ReadAllText((Join-Path $cartella 'issued-chain.pem')), '-----BEGIN CERTIFICATE-----.*?-----END CERTIFICATE-----', 'Singleline').Value
$impronta = [Security.Cryptography.X509Certificates.X509Certificate2]::CreateFromPem($leaf).Thumbprint
"Impronta del certificato di ${sensore}: $impronta"
az eventgrid namespace client create `
    --name $sensoreImpronta `
    --namespace-name $eg `
    --resource-group $rg `
    --authentication-name $sensoreImpronta `
    --client-certificate-authentication "{validationScheme:ThumbprintMatch,allowedThumbprints:[$impronta]}" `
    --description "Device registrato in ADR: $sensore, riconosciuto dall'impronta" `
    -o none `
    --only-show-errors
Assert-Az "Creazione del client $sensoreImpronta"
"$sensoreImpronta censito. Event Grid puo' metterci qualche decina di secondi a usarlo."


In [ ]:
az eventgrid namespace client list `
    --namespace-name $eg `
    --resource-group $rg `
    --query "[].{client:name, autenticazione:authenticationName, metodo:clientCertificateAuthentication.validationScheme, stato:state}" `
    -o table `
    --only-show-errors
Assert-Az 'Elenco dei client'

## Cosa abbiamo visto
- Un namespace Event Grid con **MQTT** acceso, e la **Issuing CA di ADR** caricata.
- Il device è **censito due volte**: in ADR (anagrafica e certificato) e in Event Grid (client). **Event Grid non si collega ad ADR**: i collegamenti del namespace ADR sono DPS, Hub e aggiornamenti software.
  È lo stesso problema della slide sugli inventari separati, qui per Event Grid.
- Due client per lo stesso device: **per CA** (`eventgrid-sensore-01`) e **per impronta** (`eventgrid-sensore-01-impronta`). Quale dei due accetta il certificato di ADR si vede nel notebook 21.

**Prossimo notebook: 21, il device si collega.**
